In [23]:
import pandas as pd
import numpy as np
import os
import pickle
from scipy.stats import pearsonr, spearmanr
os.chdir('/mnt/mr01-home01/m65338lb/worktrees/rnaDecay/gena_lm_extraFeatures')


In [24]:
orig = pd.read_csv('output/data/sanityCheck/originalSeqs_predictions.csv')
same = pd.read_csv('output/data/sanityCheck/sameSeq_predictions.csv')
utr5 = pd.read_csv('output/data/sanityCheck/same5UTR_predictions.csv')
utr3 = pd.read_csv('output/data/sanityCheck/same3UTR_predictions.csv')

# merge the two dataframes on sequenceCounts
merged = pd.merge(orig, same, on='sequenceCount', suffixes=('_orig', '_same'), how='outer')
merged = pd.merge(merged, utr5, on='sequenceCount', suffixes=('', '_utr5'), how='outer')
merged = pd.merge(merged, utr3, on='sequenceCount', suffixes=('', '_utr3'), how='outer')

In [25]:
merged

,actualDecayRate_orig,predictedDecayRate_orig,sequenceCount,actualDecayRate_same,predictedDecayRate_same,actualDecayRate,predictedDecayRate,actualDecayRate_utr3,predictedDecayRate_utr3
0,4.05,4.89,1,4.05,4.89,4.05,4.89,4.05,4.89
1,4.05,4.55,2,4.77,4.85,4.77,4.83,4.77,4.74
2,4.05,4.76,3,5.29,4.89,5.29,5.03,5.29,4.38
3,4.05,3.07,4,2.91,4.92,2.91,3.24,2.91,4.06
4,4.05,3.97,5,3.74,4.76,3.74,4.76,3.74,3.33
...,...,...,...,...,...,...,...,...,...
91,NaN,NaN,13,1.10,4.95,NaN,NaN,1.10,4.99
92,NaN,NaN,20,3.64,4.84,NaN,NaN,3.64,3.54
93,NaN,NaN,23,2.76,4.65,NaN,NaN,2.76,4.05
94,NaN,NaN,43,2.11,4.88,NaN,NaN,2.11,4.31


In [26]:
importance = pickle.load(open('/mnt/mr01-home01/m65338lb/worktrees/rnaDecay/gena_lm_extraFeatures/output/importance/shap.pkl', 'rb'))


In [27]:
counts = [x+1 for x in range(len(importance))]
actualPredictions = [imp.text.split(' ')[1].split(':')[1] for imp in importance]

ap = pd.DataFrame({'sequenceCount': counts, 'actualPredictions': actualPredictions})

# merge combined with ap
merged_df = pd.merge(ap, merged, on='sequenceCount', suffixes=('_ap',''), how='outer')

In [28]:
merged_df

,sequenceCount,actualPredictions,actualDecayRate_orig,predictedDecayRate_orig,actualDecayRate_same,predictedDecayRate_same,actualDecayRate,predictedDecayRate,actualDecayRate_utr3,predictedDecayRate_utr3
0,1,4.89,4.05,4.89,4.05,4.89,4.05,4.89,4.05,4.89
1,2,4.52,4.05,4.55,4.77,4.85,4.77,4.83,4.77,4.74
2,3,4.76,4.05,4.76,5.29,4.89,5.29,5.03,5.29,4.38
3,4,3.11,4.05,3.07,2.91,4.92,2.91,3.24,2.91,4.06
4,5,3.84,4.05,3.97,3.74,4.76,3.74,4.76,3.74,3.33
...,...,...,...,...,...,...,...,...,...,...
91,92,NaN,4.05,4.29,4.50,4.83,4.50,4.91,4.50,3.59
92,94,NaN,4.05,4.42,2.29,4.93,2.29,4.91,2.29,4.59
93,95,NaN,4.05,3.71,5.73,4.94,5.73,5.13,5.73,5.15
94,96,NaN,4.05,3.09,4.53,4.77,4.53,5.04,4.53,3.01


In [29]:
print(merged_df[['actualPredictions','predictedDecayRate_orig','predictedDecayRate_same','predictedDecayRate','predictedDecayRate_utr3']].to_numpy().T)

[['4.89' '4.52' '4.76' '3.11' '3.84' '3.36' '4.14' '4.19' '4.52' '5.19'
  '3.78' '4.39' '3.82' '4.71' '4.54' '4.64' '4.69' '2.84' '2.98' '4.54'
  '4.09' '4.69' '4.41' '4.79' '4.32' '3.99' '3.51' '3.52' '3.95' '4.75'
  '3.74' '2.12' '3.18' '2.73' '2.79' '4.18' '5.25' '5.47' '3.8' '3.29'
  '3.79' '4.58' '4.44' '3.31' '5.59' '4.43' '3.63' '4.0' '3.07' '2.49'
  '3.51' '3.41' '2.67' '3.04' '4.96' '4.87' '3.42' '2.08' '3.61' '5.05'
  '3.8' '5.09' '3.7' '4.09' '5.05' '2.31' '3.34' '2.78' '2.91' '2.49'
  '4.7' '4.24' '3.23' '4.01' '3.96' '5.34' '5.36' '4.4' '4.41' '4.02'
  '3.17' '3.05' '4.99' '4.82' '3.75' '2.22' '2.2' '4.23' '4.46' '3.75'
  '2.94' nan nan nan nan nan]
 [4.89 4.55 4.76 3.07 3.97 3.62 4.2 4.23 4.58 5.25 3.82 4.35 nan 3.79
  4.68 4.51 4.67 4.71 2.91 nan 3.01 4.75 nan 4.28 4.61 4.34 4.81 4.35
  4.17 3.69 3.54 3.94 5.01 3.75 2.19 3.24 2.72 2.8 4.31 5.25 5.39 3.7 nan
  3.33 3.81 4.51 4.36 3.17 5.48 4.5 3.7 3.95 3.09 2.47 3.54 3.46 2.92 3.3
  4.91 4.8 3.44 2.16 3.58 5.14 3.76 4.96 

In [30]:
# Convert 'actualPredictions' column to numeric, coercing errors to NaN
merged_df['actualPredictions'] = pd.to_numeric(merged_df['actualPredictions'], errors='coerce')

# Now calculate and print the mean of the absolute difference
print(np.mean(abs(merged_df['actualPredictions'] - merged_df['predictedDecayRate_orig'])))
print(np.mean(abs(merged_df['predictedDecayRate_orig'][0] - merged_df['predictedDecayRate_same'])))

print(np.mean(abs(merged_df['actualPredictions'] - merged_df['predictedDecayRate'])))
print(np.mean(abs(merged_df['actualPredictions'] - merged_df['predictedDecayRate_utr3'])))

print(np.mean(abs(merged_df['predictedDecayRate_orig'][0] - merged_df['predictedDecayRate'])))
print(np.mean(abs(merged_df['predictedDecayRate_orig'][0] - merged_df['predictedDecayRate_utr3'])))

0.8708045977011494
0.07635416666666668
1.0822093023255814
0.9139560439560441
0.37200000000000005
0.7675
